# Handwritten + Digital OCR — pick & save the BEST model (Kaggle)

This notebook benchmarks the off-the-shelf `microsoft/trocr-base-handwritten` against a fine-tuned
version, then **automatically saves whichever scores lower CER**. You can never accidentally ship
the worse model. The saved model drops into your existing `PDF_Extractor.py` backend and the
notebook also demos the full **digital-PDF + handwritten** routing.

### Why this design (important)
In testing, the pretrained model scored **CER 4.79%** while a 1-epoch fine-tune on a small IAM
subset scored **CER 10.94%** — fine-tuning made it *worse*. That's expected: TrOCR-base-handwritten
is **already trained on IAM**, so re-training it on IAM can only match or degrade it. **Fine-tuning
only helps once you add your OWN handwriting** that differs from IAM. This notebook therefore keeps
the pretrained model unless a fine-tune genuinely beats it on held-out data.

### Honest accuracy note
No OCR is 100% on arbitrary handwriting. ~3–7% CER on clean lines is realistic; messy/cursive/phone
photos are worse. The decisive test is your own documents, demoed in section 11.

> ⚠️ Turn **Internet → On** and **Accelerator → GPU (T4)** in the notebook settings.


## 1. Setup

In [ ]:
# Kaggle ships PyTorch + CUDA. Install the OCR/training/PDF stack.
!pip install -q -U "transformers>=4.46" "datasets>=2.19" "accelerate>=0.30" jiwer sentencepiece pymupdf
print("installs done")


In [ ]:
import torch, transformers, platform
print("python      :", platform.python_version())
print("torch       :", torch.__version__)
print("transformers:", transformers.__version__)
print("CUDA        :", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## 2. Config
`DO_FINETUNE=True` runs the comparison so you can see it for yourself; the best model is saved
either way. For a real fine-tune that has a chance of winning, set `QUICK_TEST=False` **and** add
your own labelled lines (see section 13).

In [ ]:
from pathlib import Path

BASE_MODEL     = "microsoft/trocr-base-handwritten"   # or "...-large-handwritten" (more accurate, heavier)
DATASET        = "Teklia/IAM-line"                    # train 6482 / val 976 / test 2915 -> columns: image, text
MAX_TARGET_LEN = 64

DO_FINETUNE    = True       # compare a fine-tune vs the pretrained baseline
QUICK_TEST     = True       # True = tiny/fast smoke test. Set False for the real run.
EPOCHS         = 1 if QUICK_TEST else 6
TRAIN_SUBSET   = 300 if QUICK_TEST else None          # None = full training set
EVAL_SUBSET    = 100 if QUICK_TEST else 500           # test samples scored for the comparison
BATCH_SIZE     = 8                                     # drop to 4 on OOM
LR             = 2e-5                                  # gentle: the base is already IAM-trained

OUTPUT_DIR = Path("/kaggle/working/trocr-best")        # the winning model lands here
CAND_PRE   = Path("/kaggle/working/_cand_pretrained")  # candidate folders (temp)
CAND_FT    = Path("/kaggle/working/_cand_finetuned")
for p in (OUTPUT_DIR, CAND_PRE, CAND_FT):
    p.mkdir(parents=True, exist_ok=True)
print("config ready | DO_FINETUNE =", DO_FINETUNE, "| QUICK_TEST =", QUICK_TEST)


## 3. Load the IAM handwriting dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset(DATASET)
train_ds, val_ds, test_ds = ds["train"], ds["validation"], ds["test"]
if TRAIN_SUBSET:
    train_ds = train_ds.select(range(min(TRAIN_SUBSET, len(train_ds))))
    val_ds   = val_ds.select(range(min(max(50, TRAIN_SUBSET // 5), len(val_ds))))
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")
print("example:", train_ds[0]["text"])


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 1, figsize=(12, 4))
for ax, i in zip(axes, range(3)):
    ax.imshow(train_ds[i]["image"], cmap="gray"); ax.set_title(train_ds[i]["text"], fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()


## 4. Load TrOCR + metric / inference helpers

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import jiwer

processor = TrOCRProcessor.from_pretrained(BASE_MODEL)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL).to(DEVICE).eval()

def score(preds, refs):
    preds = [p if p.strip() else " " for p in preds]
    refs  = [r if r.strip() else " " for r in refs]
    return {"CER": jiwer.cer(refs, preds) * 100, "WER": jiwer.wer(refs, preds) * 100}


In [ ]:
@torch.no_grad()
def transcribe_batch(images, num_beams=4, max_len=MAX_TARGET_LEN):
    images = [im.convert("RGB") for im in images]
    pv = processor(images=images, return_tensors="pt").pixel_values.to(DEVICE)
    gen = model.generate(pv, num_beams=num_beams, max_length=max_len)
    return processor.batch_decode(gen, skip_special_tokens=True)

def evaluate_on(dataset, n, batch=16, tag=""):
    n = min(n, len(dataset)); preds, refs = [], []
    model.eval()
    for s in range(0, n, batch):
        chunk = dataset.select(range(s, min(s + batch, n)))
        preds += transcribe_batch(chunk["image"]); refs += chunk["text"]
    m = score(preds, refs)
    print(f"[{tag}] n={n}  CER={m['CER']:.2f}%  WER={m['WER']:.2f}%")
    return m, preds, refs


## 5. Baseline — the pretrained model (and save it as a candidate)

In [ ]:
baseline_metrics, base_preds, base_refs = evaluate_on(test_ds, EVAL_SUBSET, tag="pretrained")
for p, r in zip(base_preds[:5], base_refs[:5]):
    print(f"GT  : {r}\nPRED: {p}\n")

# snapshot the pretrained weights BEFORE any fine-tuning can change them
model.save_pretrained(CAND_PRE); processor.save_pretrained(CAND_PRE)
print("pretrained candidate saved ->", CAND_PRE)


## 6. (Optional) Fine-tune — kept only if it beats the baseline
Fine-tuning on IAM alone usually **loses** to the pretrained model (it's already IAM-trained). To
make this win, set `QUICK_TEST=False` and mix in your own labelled lines. Either way the selection
in section 8 protects you.

In [ ]:
if DO_FINETUNE:
    from transformers import (GenerationConfig, Seq2SeqTrainer,
                              Seq2SeqTrainingArguments, default_data_collator)
    from torch.utils.data import Dataset

    # --- structural config training needs (NOT generation params) ---
    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.config.pad_token_id           = processor.tokenizer.pad_token_id
    model.config.eos_token_id           = processor.tokenizer.sep_token_id
    model.config.vocab_size             = model.config.decoder.vocab_size

    # --- generation params live ONLY on generation_config ---
    gc = model.generation_config
    gc.decoder_start_token_id = processor.tokenizer.cls_token_id
    gc.pad_token_id           = processor.tokenizer.pad_token_id
    gc.eos_token_id           = processor.tokenizer.sep_token_id
    gc.max_length = MAX_TARGET_LEN; gc.num_beams = 4
    gc.early_stopping = True; gc.no_repeat_ngram_size = 3

    # --- strip any legacy generation params off model.config so generate() can't error ---
    _keep = {"pad_token_id","bos_token_id","eos_token_id","decoder_start_token_id","vocab_size"}
    for _k, _d in GenerationConfig().to_dict().items():
        if _k.startswith("_") or _k == "transformers_version" or _k in _keep: continue
        if hasattr(model.config, _k):
            try: setattr(model.config, _k, _d)
            except Exception: pass

    class HTRDataset(Dataset):
        def __init__(self, split): self.ds = split
        def __len__(self): return len(self.ds)
        def __getitem__(self, i):
            row = self.ds[i]; img = row["image"].convert("RGB")
            pv = processor(images=img, return_tensors="pt").pixel_values.squeeze(0)
            lab = processor.tokenizer(row["text"], padding="max_length",
                                      max_length=MAX_TARGET_LEN, truncation=True).input_ids
            lab = [t if t != processor.tokenizer.pad_token_id else -100 for t in lab]
            return {"pixel_values": pv, "labels": torch.tensor(lab)}

    def compute_metrics(pred):
        pred_str = processor.batch_decode(pred.predictions, skip_special_tokens=True)
        lab = [[t if t != -100 else processor.tokenizer.pad_token_id for t in s] for s in pred.label_ids]
        label_str = processor.batch_decode(lab, skip_special_tokens=True)
        return {"cer": jiwer.cer([r or " " for r in label_str], [p or " " for p in pred_str])}

    args = Seq2SeqTrainingArguments(
        output_dir=str(OUTPUT_DIR / "_trainer"),
        predict_with_generate=True,
        eval_strategy="epoch",        # transformers < 4.46: rename to evaluation_strategy
        save_strategy="no",           # avoids the buggy best-model reload key-mismatch path
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS, learning_rate=LR,
        fp16=torch.cuda.is_available(), logging_steps=50, report_to="none",
    )
    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=HTRDataset(train_ds), eval_dataset=HTRDataset(val_ds),
        data_collator=default_data_collator, compute_metrics=compute_metrics,
        processing_class=processor,
    )
    trainer.train()
else:
    print("DO_FINETUNE=False — skipping training; the pretrained model will be shipped.")


## 7. Evaluate the fine-tuned candidate

In [ ]:
if DO_FINETUNE:
    finetuned_metrics, ft_preds, ft_refs = evaluate_on(test_ds, EVAL_SUBSET, tag="fine-tuned")
    model.save_pretrained(CAND_FT); processor.save_pretrained(CAND_FT)
    print("fine-tuned candidate saved ->", CAND_FT)
    print("\n--- fine-tuned sample predictions ---")
    for p, r in zip(ft_preds[:6], ft_refs[:6]):
        print(f"GT  : {r}\nPRED: {p}\n")
else:
    finetuned_metrics = None
    print("no fine-tuned candidate to evaluate")


## 8. Select & save the BEST model

In [ ]:
import shutil

pre_cer = baseline_metrics["CER"]
ft_cer  = finetuned_metrics["CER"] if (DO_FINETUNE and finetuned_metrics) else float("inf")

if ft_cer < pre_cer:
    best_src, best_name, best_cer = CAND_FT, "fine-tuned", ft_cer
else:
    best_src, best_name, best_cer = CAND_PRE, "pretrained", pre_cer

print("================ COMPARISON ================")
print(f"pretrained : CER {pre_cer:.2f}%")
print(f"fine-tuned : CER {ft_cer:.2f}%" if ft_cer != float("inf") else "fine-tuned : (not run)")
print(f"WINNER     : {best_name}  (CER {best_cer:.2f}%)")
print("============================================")

# copy the winner into OUTPUT_DIR
if OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
shutil.copytree(best_src, OUTPUT_DIR)

# reload the winner so every cell below uses the best model
model = VisionEncoderDecoderModel.from_pretrained(OUTPUT_DIR).to(DEVICE).eval()
processor = TrOCRProcessor.from_pretrained(OUTPUT_DIR)

zip_path = shutil.make_archive("/kaggle/working/trocr-best", "zip", OUTPUT_DIR)
print("BEST model saved to", OUTPUT_DIR)
print("download this from the Output panel:", zip_path)


## 9. Verify the saved model reloads and works
Reloads from disk exactly the way your backend will (`from_pretrained`) and runs inference. If you
see sensible transcriptions, the download is safe to ship.

In [ ]:
_proc  = TrOCRProcessor.from_pretrained(OUTPUT_DIR)
_model = VisionEncoderDecoderModel.from_pretrained(OUTPUT_DIR).to(DEVICE).eval()
non_empty = 0
for i in range(5):
    img = test_ds[i]["image"].convert("RGB")
    pv = _proc(images=img, return_tensors="pt").pixel_values.to(DEVICE)
    with torch.no_grad():
        out = _model.generate(pv, max_length=MAX_TARGET_LEN, num_beams=4)
    pred = _proc.batch_decode(out, skip_special_tokens=True)[0]
    print(f"GT  : {test_ds[i]['text']}\nPRED: {pred}\n")
    non_empty += int(bool(pred.strip()))
assert non_empty >= 4, "Reloaded model produced empty output — investigate before shipping."
print("OK — saved model reloads and produces text. Safe to use as backend.")


## 10. Full-page handwriting inference (line segmentation -> TrOCR)
TrOCR reads one line at a time, so a full page is split into line strips first. Projection-profile
segmentation works for clean single-column pages; for messy layouts swap in a detector (PaddleOCR
/ docTR) and feed each line crop to `transcribe_batch`.

In [ ]:
import numpy as np
from PIL import Image

def segment_lines(pil_image, min_height=12, pad=4):
    gray = np.array(pil_image.convert("L"))
    ink = (gray < gray.mean()).astype(np.uint8)
    active = ink.sum(axis=1) > (0.005 * gray.shape[1])
    strips, start, inside = [], 0, False
    for i, a in enumerate(active):
        if a and not inside: inside, start = True, i
        elif not a and inside:
            inside = False
            if i - start >= min_height:
                top, bot = max(0, start - pad), min(gray.shape[0], i + pad)
                strips.append(pil_image.crop((0, top, pil_image.width, bot)))
    if inside and gray.shape[0] - start >= min_height:
        strips.append(pil_image.crop((0, max(0, start - pad), pil_image.width, gray.shape[0])))
    return strips or [pil_image]

def recognize_full_page(pil_image):
    lines = segment_lines(pil_image)
    out = []
    for i in range(0, len(lines), 16):
        out += transcribe_batch(lines[i:i+16])
    return "\n".join(t for t in out if t.strip())

# demo: stack a few IAM lines into one 'page'
imgs = [test_ds[i]["image"].convert("L") for i in range(4)]
W = max(im.width for im in imgs)
page = Image.new("L", (W, sum(im.height for im in imgs) + 50), 255)
y = 10
for im in imgs: page.paste(im, (0, y)); y += im.height + 10
print(recognize_full_page(page))


## 11. Digital + handwritten PDF (end-to-end routing)
This mirrors your `PDF_Extractor.py`: if a page already has embedded text it's a **digital** PDF
(use the text directly); otherwise the page is rasterised and sent to **TrOCR** (handwritten /
scanned). Point `pdf_to_text` at a PDF you add as a Kaggle dataset to test both paths.

In [ ]:
import fitz  # PyMuPDF
from PIL import Image

def pdf_to_text(pdf_path, min_chars=20):
    doc = fitz.open(pdf_path); pages = []
    for i, page in enumerate(doc):
        txt = (page.get_text("text") or "").strip()
        if len(txt) >= min_chars:
            pages.append({"page": i + 1, "mode": "digital", "text": txt})
        else:
            pix = page.get_pixmap(matrix=fitz.Matrix(3, 3), alpha=False)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            pages.append({"page": i + 1, "mode": "handwritten/ocr", "text": recognize_full_page(img)})
    doc.close()
    return pages

# Example (uncomment after adding a PDF):
# for pg in pdf_to_text("/kaggle/input/your-dataset/sample.pdf"):
#     print(f"--- page {pg['page']} [{pg['mode']}] ---\n{pg['text']}\n")
print("pdf_to_text() ready — handles digital pages via embedded text and scanned pages via TrOCR.")


## 12. Backend drop-in: new `DL_OCR_Engine.py`
Keeps the exact interface your `PDF_Extractor.py` calls — `is_available()`, `get_dl_ocr_engine()`,
`engine.extract_page(image) -> {"text","confidence","tool"}` — so nothing else changes. Point
`_MODEL_DIR` at the unzipped `trocr-best/` folder.

In [ ]:
%%writefile /kaggle/working/DL_OCR_Engine.py
"""
TrOCR handwriting OCR engine — drop-in replacement for the CRNN engine.
Same public interface as the original DL_OCR_Engine.py:
    engine = get_dl_ocr_engine()
    result = engine.extract_page(pil_image)   # -> {"text","confidence","tool"}
"""
import logging
from pathlib import Path
Logger = logging.getLogger(__name__)

# Folder produced by the notebook (unzip trocr-best.zip next to this file),
# or a Hugging Face repo id like "your-username/trocr-best".
_MODEL_DIR = Path(__file__).resolve().parent / "trocr-best"

try:
    import torch
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    _deps_ok = True
except ImportError:
    _deps_ok = False
    Logger.warning("transformers/torch not available — TrOCR engine disabled")

_ENGINE_INSTANCE = None

def is_available() -> bool:
    return _deps_ok and (Path(str(_MODEL_DIR)).exists() or "/" in str(_MODEL_DIR))

def get_dl_ocr_engine():
    global _ENGINE_INSTANCE
    if _ENGINE_INSTANCE is None:
        _ENGINE_INSTANCE = _TrOCR_Engine()
    return _ENGINE_INSTANCE

class _TrOCR_Engine:
    def __init__(self, model_dir=_MODEL_DIR):
        if not _deps_ok:
            raise RuntimeError("TrOCR engine unavailable — install torch + transformers")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        src = str(model_dir)
        self.processor = TrOCRProcessor.from_pretrained(src)
        self.model = VisionEncoderDecoderModel.from_pretrained(src).to(self.device).eval()
        self.max_len = 64
        Logger.info("TrOCR engine loaded from %s on %s", src, self.device)

    def _segment_lines(self, pil_image, min_height=12, pad=4):
        import numpy as np
        gray = np.array(pil_image.convert("L"))
        active = (gray < gray.mean()).astype("uint8").sum(axis=1) > (0.005 * gray.shape[1])
        strips, start, inside = [], 0, False
        for i, a in enumerate(active):
            if a and not inside: inside, start = True, i
            elif not a and inside:
                inside = False
                if i - start >= min_height:
                    top, bot = max(0, start - pad), min(gray.shape[0], i + pad)
                    strips.append(pil_image.crop((0, top, pil_image.width, bot)))
        if inside and gray.shape[0] - start >= min_height:
            strips.append(pil_image.crop((0, max(0, start - pad), pil_image.width, gray.shape[0])))
        return strips or [pil_image]

    @torch.no_grad()
    def _read_lines(self, images):
        images = [im.convert("RGB") for im in images]
        pix = self.processor(images=images, return_tensors="pt").pixel_values.to(self.device)
        out = self.model.generate(pix, max_length=self.max_len, num_beams=4,
                                  output_scores=True, return_dict_in_generate=True)
        texts = self.processor.batch_decode(out.sequences, skip_special_tokens=True)
        try:
            confs = [float(torch.exp(s)) for s in out.sequences_scores]
        except (AttributeError, TypeError):
            confs = [0.0] * len(texts)
        return texts, confs

    def extract_page(self, image) -> dict:
        try:
            lines = self._segment_lines(image)
            texts, confs = [], []
            for i in range(0, len(lines), 16):
                t, c = self._read_lines(lines[i:i + 16])
                for txt, cf in zip(t, c):
                    if txt.strip(): texts.append(txt.strip()); confs.append(cf)
            avg = round(sum(confs) / len(confs) * 100, 2) if confs else 0.0
            return {"text": "\n".join(texts), "confidence": avg, "tool": "trocr_handwritten"}
        except Exception as exc:
            Logger.warning("TrOCR page extraction failed: %s", exc)
            return {"text": "", "confidence": 0.0, "tool": "trocr_handwritten"}


## 13. Wiring it into your backend

1. Download **`trocr-best.zip`** from the Kaggle Output panel; unzip so `trocr-best/` sits next to
   your Python files.
2. Replace your old `DL_OCR_Engine.py` with the one written in section 12 (also downloadable from
   `/kaggle/working/DL_OCR_Engine.py`).
3. `pip install torch transformers pymupdf` on your server. **`PDF_Extractor.py` is unchanged** — it
   still calls `get_dl_ocr_engine().extract_page(image)` and gets `{"text","confidence","tool"}`.
   Digital PDFs use PyMuPDF's embedded text; scanned/handwritten pages fall through to TrOCR — exactly
   the routing demoed in section 11.

### To make fine-tuning actually win (when you need it)
- Add **your own labelled lines** (image + ground-truth text) to `train_ds`. Even 200–500 of your
  real samples mixed with IAM shifts the model toward your handwriting — that's the only way a
  fine-tune beats the IAM-pretrained baseline.
- Set `QUICK_TEST=False`, keep `LR=2e-5`, raise `EPOCHS`.
- Try `BASE_MODEL="microsoft/trocr-large-handwritten"` for a few points more accuracy.
- For messy page layouts, replace `segment_lines` with a real detector (PaddleOCR / docTR).
